# Hydro Data: Load Selected Variables and Spatial Ranges

This notebook provides a comprehensive guide to selective data loading and spatial filtering in Mera.jl. You'll learn advanced techniques for efficiently loading only the data you need from large hydrodynamic simulations.

## Learning Objectives

- Master selective variable loading for memory optimization
- Apply spatial filtering and region selection techniques
- Work with different coordinates and units
- Understand center-relative coordinate systems
- Optimize data loading for large simulations

## Quick Reference: Data Selection Functions

This section provides a comprehensive reference of Mera.jl functions for selective data loading and spatial filtering.

### Variable Selection
```julia
# Load all variables (default behavior)
gas = gethydro(info)

# Select specific variables by name
gas = gethydro(info, vars=[:rho, :p, :vx])            # Density, pressure, x-velocity
gas = gethydro(info, vars=[:var1, :var5, :var2])      # Using variable numbers

# Select variables without keyword (order matters: info, variables)
gas = gethydro(info, [:rho, :p])                      # Multiple variables
gas = gethydro(info, :vx)                             # Single variable

# Common variable names and numbers
# :varn1 or :cpu  → CPU number (= -1)
# :var1 or :rho   → Density
# :var2 or :vx    → X-velocity  
# :var3 or :vy    → Y-velocity
# :var4 or :vz    → Z-velocity
# :var5 or :p     → Pressure
```

### Spatial Range Selection
```julia
# RAMSES standard notation (domain: [0:1]³)
gas = gethydro(info, xrange=[0.2, 0.8],              # X-range filter
                     yrange=[0.2, 0.8],              # Y-range filter  
                     zrange=[0.4, 0.6])              # Z-range filter

# Center-relative coordinates (RAMSES units)
gas = gethydro(info, xrange=[-0.3, 0.3],             # Relative to center
                     yrange=[-0.3, 0.3],
                     zrange=[-0.1, 0.1],
                     center=[0.5, 0.5, 0.5])

# Physical units (e.g., kpc)
gas = gethydro(info, xrange=[2., 22.],                # Physical coordinates
                     yrange=[2., 22.],
                     zrange=[22., 26.],
                     range_unit=:kpc)

# Center-relative with physical units
gas = gethydro(info, xrange=[-16., 16.],              # Relative to center in kpc
                     yrange=[-16., 16.],
                     zrange=[-2., 2.],
                     center=[24., 24., 24.],
                     range_unit=:kpc)

# Box center shortcuts
gas = gethydro(info, center=[:boxcenter])            # All dimensions centered
gas = gethydro(info, center=[:bc])                   # Short form
gas = gethydro(info, center=[:bc, 24., :bc])         # Mixed: center x,z; fixed y
```

### Performance Optimization
```julia
# Limit refinement levels for faster loading
gas = gethydro(info, lmax=8)                         # Maximum level 8 (existing higher levels are scaled down)

# Combined optimizations
gas = gethydro(info, [:rho, :p],                     # Select variables
                     lmax=10,                        # Limit levels
                     xrange=[-10., 10.],             # Spatial range
                     yrange=[-10., 10.],
                     zrange=[-2., 2.],
                     center=[:bc],                   # Box center
                     range_unit=:kpc)                # Physical units
```

### Available Physical Units
```julia
# Check available units in simulation
viewfields(info.scale)

# Common length units
:m, :km, :cm, :mm, :μm, :Mpc, :kpc, :pc, :ly, :au, :Rsun
```

## Getting Started: Simulation Setup

Before exploring data selection techniques, let's load our simulation and examine its properties. This establishes the foundation for all subsequent data loading operations.

In [1]:
# Example-data root. Point this at your own simulation folder, or set the
# MERA_EXAMPLES environment variable; every path below is built from it.
MERA_EXAMPLES = get(ENV, "MERA_EXAMPLES", "/Volumes/FASTStorage/Simulations/Mera-Tests");

using Mera
info = getinfo(300, "$MERA_EXAMPLES/RAMSES/mw_L10");


*__   __ _______ ______   _______ 


|  |_|  |       |    _ | |   _   |
|       |    ___|   | || |  |_|  |
|       |   |___|   |_||_|       |
|       |    ___|    __  |       |
| ||_|| |   |___|   |  | |   _   |
|_|   |_|_______|___|  |_|__| |__|
Mera v1.8.0

[Mera]: 2026-08-05T15:58:21.377



Code: RAMSES


output [300] summary:
mtime: 2023-04-09T05:34:09
ctime: 2025-06-21T18:31:24.020
simulation time: 445.89 [Myr]
boxlen: 48.0 [kpc]
ncpu: 640
ndim: 3
cosmological:  false
-------------------------------------------------------
amr:           true
level(s): 6 - 10 --> cellsize(s): 750.0 [pc] - 46.88 [pc]
-------------------------------------------------------
hydro:         true
hydro-variables:  

7  --> (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01)
hydro-descriptor: (:density, :velocity_x, :velocity_y, :velocity_z, :pressure, :scalar_00, :scalar_01)
γ: 1.6667
-------------------------------------------------------
gravity:       true
gravity-variables: (:epot, :ax, :ay, :az)
-------------------------------------------------------
particles:     true
- Nstars:   5.445150e+05 
particle-variables: 

7  --> (:vx, :vy, :vz, :mass, :family, :tag, :birth)
particle-descriptor: (:position_x, :position_y, :position_z, :velocity_x, :velocity_y, :velocity_z, :mass, :identity, :levelp, :family, :tag, :birth_time)
-------------------------------------------------------
rt:            false
clumps:           false
-------------------------------------------------------
namelist-file: 

("&COOLING_PARAMS", "&SF_PARAMS", "&AMR_PARAMS", "&BOUNDARY_PARAMS", "&OUTPUT_PARAMS", "&POISSON_PARAMS", "&RUN_PARAMS", "&FEEDBACK_PARAMS", "&HYDRO_PARAMS", "&INIT_PARAMS", "&REFINE_PARAMS")
-------------------------------------------------------
timer-file:       true
compilation-file: false
makefile:         true
patchfile:        true



## Variable Selection Techniques

Understanding how to selectively load variables is crucial for efficient memory usage and faster analysis. Mera provides flexible approaches to variable selection, from loading everything to precise variable targeting.

### Understanding Variable References

Mera provides flexible ways to reference hydrodynamic variables. Understanding these reference methods enables precise control over data loading.

**Core Variable References:**

| Variable | Symbol Format | Number Format | Description |
|----------|---------------|---------------|-------------|
| CPU Number | `:cpu` | `:varn1` | Processor identification (= -1) |
| Density | `:rho` | `:var1` | Mass density |
| X-Velocity | `:vx` | `:var2` | Velocity component in x-direction |
| Y-Velocity | `:vy` | `:var3` | Velocity component in y-direction |
| Z-Velocity | `:vz` | `:var4` | Velocity component in z-direction |
| Pressure | `:p` | `:var5` | Gas pressure |
| Additional | - | `:var6`, `:var7`, ... | Extended variables |

**Key Features:**
- Variable order is flexible in function calls
- Both symbolic (`:rho`) and numeric (`:var1`) formats supported
- Future updates will support descriptor file variable names
- Consistent naming across all Mera hydro functions

### Loading All Variables (Default Behavior)

The simplest approach is to load all available variables. This is the default behavior when no specific variables are requested.

In [2]:
gas = gethydro(info);

[Mera]: Get hydro data: 2026-08-05T15:58:25.005



Key vars=(:level, :cx, :cy, :cz)


Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

domain:


xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:


   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



...

  Threading: 4 threads for 11 columns


  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads


  Creating IndexedTable with 11 columns...
✓ Table created in 41.775 seconds


Memory used for data table :2.321086215786636

 GB
-------------------------------------------------------



In [3]:
gas.data

Table with 28320979 rows, 11 columns:
Columns:
#   colname    type
──────────────────────
1   level      Int64
2   cx         Int64
3   cy         Int64
4   cz         Int64
5   rho        Float64
6   vx         Float64
7   vy         Float64
8   vz         Float64
9   p          Float64
10  scalar_00  Float64
11  scalar_01  Float64

### Selecting Multiple Variables

Mera provides multiple ways to select specific variables. You can use keyword arguments or positional arguments with flexible syntax.

In [4]:
gas_a = gethydro(info, vars=[:rho, :p]); 

[Mera]: Get hydro data: 2026-08-05T15:59:27.252

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 5) = (:rho, :p) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   0%|▎                                                 |  ETA: 0:08:02 ( 0.76  s/it)

Processing files:   1%|▌                                                 |  ETA: 0:04:15 ( 0.40  s/it)

Processing files:   2%|▊                                                 |  ETA: 0:02:40 ( 0.25  s/it)

...

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:20 (31.94 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 2 variables
Creating Table from 28320979 cells with max 4 threads...
  Threading: 4 threads for 6 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads


  Creating IndexedTable with 6 columns...
✓ Table created in 2.145 seconds


Memory used for data table :1.2660471182316542 GB
-------------------------------------------------------



**Alternative:** Use variable numbers instead of symbolic names. This approach provides identical functionality:

In [5]:
gas_a = gethydro(info, vars=[:var1, :var5]); 

[Mera]: Get hydro data: 2026-08-05T15:59:50.592

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 5) = (:rho, :p) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   0%|▎                                                 |  ETA: 0:01:03 (98.63 ms/it)

Processing files:   1%|▌                                                 |  ETA: 0:00:46 (73.12 ms/it)

Processing files:   2%|▊                                                 |  ETA: 0:00:36 (56.57 ms/it)

...

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:18 (28.53 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 2 variables
Creating Table from 28320979 cells with max 4 threads...
  Threading: 4 threads for 6 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads


  Creating IndexedTable with 6 columns...
✓ Table created in 1.796 seconds


Memory used for data table :1.2660471182316542 GB
-------------------------------------------------------



**Keyword-free syntax:** When following the specific order (InfoType object, then variables), keyword arguments are optional:

In [6]:
gas_a = gethydro(info, [:rho, :p]); 

[Mera]: Get hydro data: 2026-08-05T16:00:11.305

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 5) = (:rho, :p) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   0%|▎                                                 |  ETA: 0:01:03 ( 0.10  s/it)

Processing files:   1%|▋                                                 |  ETA: 0:00:39 (61.93 ms/it)

Processing files:   2%|█                                                 |  ETA: 0:00:32 (50.71 ms/it)

...

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:18 (29.15 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 2 variables
Creating Table from 28320979 cells with max 4 threads...
  Threading: 4 threads for 6 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads


  Creating IndexedTable with 6 columns...
✓ Table created in 1.749 seconds


Memory used for data table :1.2660471182316542 GB
-------------------------------------------------------



In [7]:
gas_a.data

Table with 28320979 rows, 6 columns:
level  cx   cy   cz   rho          p
─────────────────────────────────────────────
6      1    1    1    3.18647e-9   1.06027e-9
6      1    1    2    3.58591e-9   1.33677e-9
6      1    1    3    3.906e-9     1.58181e-9
6      1    1    4    4.27441e-9   1.93168e-9
6      1    1    5    4.61042e-9   2.37842e-9
6      1    1    6    4.83977e-9   2.8197e-9
6      1    1    7    4.974e-9     3.20883e-9
6      1    1    8    5.08112e-9   3.56075e-9
6      1    1    9    5.20596e-9   3.89183e-9
6      1    1    10   5.38372e-9   4.20451e-9
6      1    1    11   5.67209e-9   4.50256e-9
6      1    1    12   6.14423e-9   4.78595e-9
⋮
10     814  493  514  0.000321702  2.18179e-6
10     814  494  509  1.42963e-6   3.35949e-6
10     814  494  510  1.4351e-6    3.38092e-6
10     814  494  511  0.00029515   2.55696e-6
10     814  494  512  0.000395273  2.5309e-6
10     814  494  513  0.000321133  2.16472e-6
10     814  494  514  0.000319678  2.17348e-6
10    

### Selecting Single Variables

For single variable selection, arrays and keywords are unnecessary. Maintain the order: InfoType object, then variable symbol:

In [8]:
gas_c = gethydro(info, :vx ); 

[Mera]: Get hydro data: 2026-08-05T16:00:32.408

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(2,) = (:vx,) 

domain:
xmin::xmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
ymin::ymax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]
zmin::zmax: 0.0 :: 1.0  	==> 0.0 [kpc] :: 48.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   0%|▎                                                 |  ETA: 0:01:04 ( 0.10  s/it)

Processing files:   1%|▌                                                 |  ETA: 0:00:45 (71.51 ms/it)

Processing files:   1%|▊                                                 |  ETA: 0:00:40 (63.12 ms/it)

...


✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 28320979 cells, 1 variables
Creating Table from 28320979 cells with max 4 threads...
  Threading: 4 threads for 5 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads


  Creating IndexedTable with 5 columns...
✓ Table created in 2.716 seconds


Memory used for data table :1.0550392987206578

 GB
-------------------------------------------------------



In [9]:
gas_c.data

Table with 28320979 rows, 5 columns:
level  cx   cy   cz   vx
────────────────────────────────
6      1    1    1    -1.25532
6      1    1    2    -1.23262
6      1    1    3    -1.2075
6      1    1    4    -1.16462
6      1    1    5    -1.10493
6      1    1    6    -1.02686
6      1    1    7    -0.948004
6      1    1    8    -0.879731
6      1    1    9    -0.824484
6      1    1    10   -0.782768
6      1    1    11   -0.754141
6      1    1    12   -0.737723
⋮
10     814  493  514  0.268398
10     814  494  509  0.00398492
10     814  494  510  0.00496945
10     814  494  511  0.303842
10     814  494  512  0.305647
10     814  494  513  0.266079
10     814  494  514  0.26508
10     814  495  511  0.289612
10     814  495  512  0.290753
10     814  496  511  0.285209
10     814  496  512  0.285463

## Spatial Range Selection Techniques

Spatial filtering is essential for focusing analysis on specific regions of interest. Mera offers multiple coordinate systems and reference methods to accommodate different analysis needs.

**Available Coordinate Systems:**
- **RAMSES Standard:** Normalized domain [0:1]³ 
- **Center-Relative:** Coordinates relative to specified points
- **Physical Units:** Real astronomical units (kpc, pc, etc.)
- **Box-Centered:** Convenient shortcuts for simulation center

This flexibility allows precise region selection for targeted analysis while optimizing memory usage and computational efficiency.

### RAMSES Standard Coordinate System

The RAMSES standard provides a normalized coordinate system that simplifies numerical calculations and ensures consistency across different simulation scales.

**Coordinate System Properties:**
- **Domain Range:** [0:1]³ in all dimensions
- **Origin:** Located at [0., 0., 0.]
- **Benefits:** Scale-independent, numerically stable
- **Usage:** Ideal for relative positioning and grid calculations

**Performance Optimization:** Use `lmax` to limit maximum refinement levels for faster loading and preview analysis. This example demonstrates level 8 restriction:

In [10]:
gas = gethydro(info, lmax=8, 
                xrange=[0.2,0.8], 
                yrange=[0.2,0.8], 
                zrange=[0.4,0.6]); 

[Mera]: Get hydro data: 2026-08-05T16:00:54.452

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

domain:


xmin::xmax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
ymin::ymax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
zmin::zmax: 0.4 :: 0.6  	==> 19.2 [kpc] :: 28.8 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   1%|▍                                                 |  ETA: 0:00:33 (52.04 ms/it)

Processing files:   1%|▋                                                 |  ETA: 0:00:29 (46.25 ms/it)

...

Processing files:  99%|█████████████████████████████████████████████████▊|  ETA: 0:00:00 (15.62 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:10 (15.68 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 1233232 cells, 7 variables
Creating Table from 1233232 cells with max 4 threads...
  Threading: 4 threads for 11 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads
  Creating IndexedTable with 11 columns...
✓ Table created in 1.81 seconds


Memory used for data table :103.4980878829956

 MB
-------------------------------------------------------



**Range Verification:** The loaded data ranges are stored in the `ranges` field using RAMSES standard notation (domain: [0:1]³):

In [11]:
gas.ranges

6-element Vector{Float64}:
 0.2
 0.8
 0.2
 0.8
 0.4
 0.6

### Center-Relative Coordinate Selection

Define spatial ranges relative to a specified center point. This approach is particularly useful for analyzing regions around specific features or objects:

In [12]:
gas = gethydro(info, lmax=8, 
                xrange=[-0.3, 0.3], 
                yrange=[-0.3, 0.3], 
                zrange=[-0.1, 0.1], 
                center=[0.5, 0.5, 0.5]); 

[Mera]: Get hydro data: 2026-08-05T16:01:06.725

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
ymin::ymax: 0.2 :: 0.8  	==> 9.6 [kpc] :: 38.4 [kpc]
zmin::zmax: 0.4 :: 0.6  	==> 19.2 [kpc] :: 28.8 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   0%|                                                  |  ETA: N/A (  N/A  s/it)

Processing files:   1%|▍                                                 |  ETA: 0:00:34 (54.28 ms/it)

Processing files:   1%|▊                                                 |  ETA: 0:00:26 (41.70 ms/it)

Processing files:   2%|█                                                 |  ETA: 0:00:24 (38.09 ms/it)

...

Processing files:  98%|█████████████████████████████████████████████████▎|  ETA: 0:00:00 ( 5.67 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:03 ( 5.69 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!


Final data size: 1233232 cells, 7 variables
Creating Table from 1233232 cells with max 4 threads...
  Threading: 4 threads for 11 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads
  Creating IndexedTable with 11 columns...
✓ Table created in 1.79 seconds


Memory used for data table :103.4980878829956 MB
-------------------------------------------------------



### Physical Unit Coordinate System

Working with physical units provides intuitive scale references for astronomical analysis. This system automatically handles unit conversions and maintains physical meaning.

**Key Advantages:**
- **Intuitive Scaling:** Use familiar astronomical units (kpc, pc, Mpc)
- **Automatic Conversion:** Mera handles unit transformations internally
- **Reference Point:** Coordinates measured from box corner [0., 0., 0.]
- **Flexibility:** Mix different units as needed for analysis

The following example demonstrates kiloparsec (kpc) coordinate selection:

In [13]:
gas = gethydro(info, lmax=8, 
                xrange=[2.,22.], 
                yrange=[2.,22.], 
                zrange=[22.,26.], 
                range_unit=:kpc); 

[Mera]: Get hydro data: 2026-08-05T16:01:12.278



Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

domain:
xmin::xmax: 0.0416667 :: 0.4583333  	==> 2.0 [kpc] :: 22.0 [kpc]
ymin::ymax: 0.0416667 :: 0.4583333  	==> 2.0 [kpc] :: 22.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   5%|██▍                                               |  ETA: 0:00:02 ( 3.64 ms/it)

Processing files:  10%|█████▏                                            |  ETA: 0:00:02 ( 3.20 ms/it)

Processing files:  16%|███████▉                                          |  ETA: 0:00:02 ( 3.14 ms/it)

...

Processing files:  98%|█████████████████████████████████████████████████▏|  ETA: 0:00:00 ( 3.21 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:02 ( 3.24 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 229992 cells, 7 variables
Creating Table from 229992 cells with max 4 threads...
  Threading: 4 threads for 11 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads


  Creating IndexedTable with 11 columns...
✓ Table created in 0.31 seconds


Memory used for data table :19.302836418151855 MB
-------------------------------------------------------



**Available Physical Units:** The `range_unit` keyword accepts various length units defined in the simulation's `scale` field:

In [14]:
viewfields(info.scale)  # or e.g.: gas.info.scale


[Mera]: Fields to scale from user/code units to selected units
Mpc

	= 0.0010000000000006482
kpc	= 1.0000000000006481
pc	= 1000.0000000006482
mpc	= 1.0000000000006482e6
ly	= 3261.5637769461323
Au	= 2.0626480623310105e23
km	= 3.0856775812820004e16
m	= 3.085677581282e19
cm	= 3.085677581282e21
mm	= 3.085677581282e22
μm	= 3.085677581282e25
Mpc3	= 1.0000000000019446e-9
kpc3	= 1.0000000000019444
pc3	= 1.0000000000019448e9
mpc3	= 1.0000000000019446e18
ly3	= 3.469585750743794e10
Au3	= 8.775571306099254e69
km3	= 2.9379989454983075e49
m3	= 2.9379989454983063e58
cm3	= 2.9379989454983065e64
mm3	= 2.937998945498306e67
μm3	= 2.937998945498306e76
Msol_pc3	= 0.9997234790001649
Msun_pc3	= 0.9997234790001649
g_cm3	= 6.76838218451376e-23
Msol_pc2	= 999.7234790008131
Msun_pc2	= 999.7234790008131
g_cm2	= 0.20885045168302602
Gyr	= 0.014910986463557083
Myr	= 14.910986463557084
yr	= 1.4910986463557083e7
s	= 4.70554946422349e14
ms	= 4.70554946422349e17
Msol	= 9.99723479002109e8
Msun	= 9.99723479002109e8
Mearth	= 3.329677459032007e14
Mjupiter	= 1.0476363431814971e12
g	= 1.98854

**Center-Relative with Physical Units:** Combine center-relative positioning with physical unit specifications for precise regional analysis:

In [15]:
gas = gethydro(info, lmax=8, 
                xrange=[-16.,16.], 
                yrange=[-16.,16.], 
                zrange=[-2.,2.], 
                center=[24.,24.,24.], 
                range_unit=:kpc); 

[Mera]: Get hydro data: 2026-08-05T16:01:14.846

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   5%|██▍                                               |  ETA: 0:00:02 ( 3.51 ms/it)

Processing files:  10%|████▊                                             |  ETA: 0:00:02 ( 3.41 ms/it)

Processing files:  15%|███████▌                                          |  ETA: 0:00:02 ( 3.28 ms/it)

Processing files:  20%|██████████▏                                       |  ETA: 0:00:02 ( 3.20 ms/it)

...

Processing files:  92%|██████████████████████████████████████████████▏   |  ETA: 0:00:00 ( 3.34 ms/it)

Processing files:  97%|████████████████████████████████████████████████▋ |  ETA: 0:00:00 ( 3.35 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:02 ( 3.40 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 7 variables
Creating Table from 650848 cells with max 4 threads...
  Threading: 4 threads for 11 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads
  Creating IndexedTable with 11 columns...
✓ Table created in 0.906 seconds


Memory used for data table :54.622477531433105 MB
-------------------------------------------------------



### Box Center Coordinate Shortcuts

Mera provides convenient shortcuts for box-centered coordinate systems, simplifying analysis focused on the simulation center.

**Available Shortcuts:**
- `:bc` or `:boxcenter` - Center coordinate for all dimensions  
- Can be applied to individual dimensions selectively
- Combines seamlessly with physical units and range specifications
- Ideal for symmetric analysis around simulation center

**Benefits:**
- Eliminates manual center calculation
- Ensures precise geometric centering
- Simplifies symmetric region definitions
- Reduces coordinate specification errors

In [16]:
gas = gethydro(info, lmax=8, 
                xrange=[-16., 16.], 
                yrange=[-16., 16.], 
                zrange=[-2., 2.], 
                center=[:boxcenter], 
                range_unit=:kpc); 

[Mera]: Get hydro data: 2026-08-05T16:01:18.086



Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

center: [0.5, 0.5, 0.5] 

==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   5%|██▍                                               |  ETA: 0:00:02 ( 3.57 ms/it)

Processing files:  10%|████▊                                             |  ETA: 0:00:02 ( 3.42 ms/it)

...

Processing files:  91%|█████████████████████████████████████████████▊    |  ETA: 0:00:00 ( 3.29 ms/it)

Processing files:  97%|████████████████████████████████████████████████▎ |  ETA: 0:00:00 ( 3.30 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:02 ( 3.36 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 7 variables
Creating Table from 650848 cells with max 4 threads...
  Threading: 4 threads for 11 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads
  Creating IndexedTable with 11 columns...
✓ Table created in 0.938 seconds


Memory used for data table :54.622477531433105 MB
-------------------------------------------------------



In [17]:
gas = gethydro(info, lmax=8, 
                xrange=[-16., 16.], 
                yrange=[-16., 16.], 
                zrange=[-2., 2.], 
                center=[:bc], 
                range_unit=:kpc); 

[Mera]: Get hydro data: 2026-08-05T16:01:21.375

Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   4%|██▎                                               |  ETA: 0:00:02 ( 3.63 ms/it)

Processing files:   9%|████▌                                             |  ETA: 0:00:02 ( 3.58 ms/it)

Processing files:  14%|██████▊                                           |  ETA: 0:00:02 ( 3.56 ms/it)

Processing files:  19%|█████████▍                                        |  ETA: 0:00:02 ( 3.45 ms/it)

...

Processing files:  95%|███████████████████████████████████████████████▎  |  ETA: 0:00:00 ( 3.30 ms/it)

Processing files:  99%|█████████████████████████████████████████████████▊|  ETA: 0:00:00 ( 3.35 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:02 ( 3.37 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 7 variables
Creating Table from 650848 cells with max 4 threads...
  Threading: 4 threads for 11 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads
  Creating IndexedTable with 11 columns...
✓ Table created in 0.945 seconds


Memory used for data table :54.622477531433105 MB
-------------------------------------------------------



**Selective Dimension Centering:** Apply box center notation to specific dimensions while maintaining explicit coordinates for others. This example centers x and z dimensions while fixing y at 24 kpc:

In [18]:
gas = gethydro(info, lmax=8, 
                xrange=[-16., 16.], 
                yrange=[-16., 16.], 
                zrange=[-2., 2.], 
                center=[:bc, 24., :bc], 
                range_unit=:kpc); 

[Mera]: Get hydro data: 2026-08-05T16:01:24.610



Key vars=(:level, :cx, :cy, :cz)
Using var(s)=(1, 2, 3, 4, 5, 6, 7) = (:rho, :vx, :vy, :vz, :p, :scalar_00, :scalar_01) 

center: [0.5, 0.5, 0.5] ==> [24.0 [kpc] :: 24.0 [kpc] :: 24.0 [kpc]]

domain:
xmin::xmax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
ymin::ymax: 0.1666667 :: 0.8333333  	==> 8.0 [kpc] :: 40.0 [kpc]
zmin::zmax: 0.4583333 :: 0.5416667  	==> 22.0 [kpc] :: 26.0 [kpc]

📊 Processing Configuration:
   Total CPU files available: 640
   Files to be processed: 640
   Compute threads: 4
   GC threads: 4



Processing files:   5%|██▎                                               |  ETA: 0:00:02 ( 3.45 ms/it)

Processing files:   9%|████▋                                             |  ETA: 0:00:02 ( 3.48 ms/it)

Processing files:  15%|███████▍                                          |  ETA: 0:00:02 ( 3.26 ms/it)

...

Processing files:  97%|████████████████████████████████████████████████▋ |  ETA: 0:00:00 ( 3.25 ms/it)

Processing files: 100%|██████████████████████████████████████████████████| Time: 0:00:02 ( 3.33 ms/it)



✓ File processing complete! Combining results...
✓ Data combination complete!
Final data size: 650848 cells, 7 variables
Creating Table from 650848 cells with max 4 threads...
  Threading: 4 threads for 11 columns
  Max threads requested: 4
  Available threads: 4
  Using parallel processing with 4 threads


  Creating IndexedTable with 11 columns...
✓ Table created in 0.905 seconds


Memory used for data table :54.622477531433105 MB
-------------------------------------------------------



## Summary

This notebook demonstrated comprehensive data selection techniques in Mera.jl, covering both variable selection and spatial filtering strategies. Key concepts covered include:

### Variable Selection Mastery
- **Flexible Reference Systems:** Using both symbolic (`:rho`) and numeric (`:var1`) variable references
- **Selective Loading:** Choosing specific variables to optimize memory usage  
- **Syntax Variations:** Keyword and positional argument approaches for different coding styles
- **Single vs. Multiple Variables:** Appropriate syntax for different selection scenarios

### Spatial Filtering Expertise  
- **Coordinate Systems:** RAMSES standard, physical units, center-relative, and box-centered approaches
- **Performance Optimization:** Using `lmax` restrictions and tight spatial bounds
- **Unit Flexibility:** Working with various astronomical length scales
- **Center Definitions:** Absolute positioning and relative coordinate systems

### Advanced Techniques
- **Combined Selection:** Integrating variable selection with spatial filtering
- **Memory Management:** Balancing analysis needs with computational resources
- **Coordinate Shortcuts:** Using box center notation for simplified positioning
- **Quality Assurance:** Verifying loaded data ranges and dimensions